# 007 — Basic RAG Pipeline (End-to-End)
**Retrieval Series** | Load → Split → Embed → Store → Retrieve → Generate

The complete RAG pipeline in a single notebook.
Every advanced technique builds on top of this foundation.


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── STEP 1: Load & Split ─────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Use all 5 downloaded articles
raw_docs = {
    "ai":  ai_text,
    "ml":  ml_text,
    "nlp": nlp_text,
    "dl":  dl_text,
    "rag": rag_text,
}

all_docs = []
for source, text in raw_docs.items():
    for chunk in splitter.split_text(text[:20000]):
        all_docs.append(Document(page_content=chunk, metadata={"source": source}))

print(f"✓ {len(all_docs)} chunks from {len(raw_docs)} sources")


In [ ]:
# ── STEP 2: Embed & Store ────────────────────────────────────────────────
import time
from langchain_community.vectorstores import FAISS

t0 = time.time()
vectorstore = FAISS.from_documents(all_docs, embeddings)
print(f"✓ Vectorstore built in {time.time()-t0:.1f}s with {len(all_docs)} vectors")

# Save to disk for reuse
import os
vs_path = os.path.join(os.getcwd(), '..', 'datasets', 'basic_rag_index')
vectorstore.save_local(vs_path)
print(f"✓ Saved to {vs_path}")


In [ ]:
# ── STEP 3: Build Retriever ──────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="mmr",          # Maximum Marginal Relevance → diverse results
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.7}
)
print("✓ MMR retriever ready  (k=5, diversity λ=0.7)")

# Quick test
test_results = retriever.invoke("What is reinforcement learning?")
print(f"\nRetrieved {len(test_results)} docs:")
for r in test_results:
    print(f"  [{r.metadata['source']}] {r.page_content[:100]}...")


In [ ]:
# ── STEP 4: RAG Chain ────────────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata['source']}]\n{d.page_content}" for d in docs
    )

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful assistant. Answer the question using ONLY the provided context.\n"
        "If the context does not contain enough information, say so.\n"
        "Always cite the source in brackets, e.g. [ai], [ml].\n\n"
        "Context:\n{context}"
    )),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)
print("✓ RAG chain assembled")


In [ ]:
# ── STEP 5: Ask Questions ─────────────────────────────────────────────────
questions = [
    "What is the difference between supervised and unsupervised learning?",
    "How does attention mechanism work in transformers?",
    "What is RAG and why is it used with LLMs?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-" * 60)


In [ ]:
# ── STEP 6: Source attribution ───────────────────────────────────────────
def rag_with_sources(question: str):
    docs = retriever.invoke(question)
    sources = list({d.metadata["source"] for d in docs})
    answer  = rag_chain.invoke(question)
    return {"answer": answer, "sources": sources, "doc_count": len(docs)}

result = rag_with_sources("What role does backpropagation play in deep learning?")
print(f"Answer  : {result['answer'][:400]}")
print(f"Sources : {result['sources']}")
print(f"Docs used: {result['doc_count']}")


## Key Takeaways
- **Load → Split → Embed → Store → Retrieve → Generate** is the core loop
- MMR retrieval gives more **diverse** results vs pure similarity
- Always include source attribution — it lets users verify answers
- Next steps: swap FAISS for hybrid (006), add reranking (009), add self-critique (010)
